<a href="https://colab.research.google.com/github/aashwill/tokens-to-megawatts/blob/main/day3_cleaning_merges_and_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 3: Cleaning a messy trace, merges and sanity checks

**Tokens to Megawatts, Layer 0, Day 3** · about 55 minutes (plus the Kaggle lessons below)

Real traces are messy: duplicate rows, inconsistent labels, missing values, impossible numbers, odd units. Today you clean a deliberately messy job trace, join it to lookup tables without silently breaking it, and finish with a set of automatic checks you'll reuse on the real Alibaba data. Same rules: one cell at a time, **predict before you run**, try the ✏️ exercises before the solutions at the bottom.

**Before this notebook (about 35 min):** Kaggle Learn *Pandas*, lessons 4–6 (Grouping and Sorting · Data Types and Missing Values · Renaming and Combining). Keep jotting the SQL equivalent next to each new command.

| Part | Topic | Time |
|---|---|---|
| 1 | First look: profile the data before touching it | 6 min |
| 2 | Duplicates | 5 min |
| 3 | Messy text labels | 6 min |
| 4 | Missing values, units, GPU-hours | 9 min |
| 5 | Flag bad rows instead of deleting them | 7 min |
| 6 | Merges: fan-out, `validate`, anti-joins, UNION ALL | 10 min |
| 7 | A reusable sanity-check function | 8 min |
| 8 | Four questions and wrap-up | 4 min |

---
## Setup: a messy job trace (synthetic)
About 600 GPU jobs over two weeks, stored the way cluster traces often are: times in **seconds since the trace started**, and GPU requests as **percent of one GPU** (100 = one GPU, 50 = half a GPU, 800 = eight GPUs). Then the setup breaks it on purpose. You don't need to follow every line here; just read the comments about what gets broken.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(seed=11)
n = 600
DAY = 24 * 3600

status = rng.choice(["Terminated", "Failed", "Running"], n, p=[0.80, 0.12, 0.08])
submit = rng.integers(0, 14 * DAY, n)
submit = np.where(status == "Running", rng.integers(int(13.5 * DAY), 14 * DAY, n), submit)   # running jobs started in the last 12 hours
wait = rng.exponential(900, n) + np.where(rng.random(n) < 0.05, rng.uniform(2, 10, n) * 3600, 0)  # 5% sit in a long queue
start = submit + wait.round()
end = start + rng.exponential(4 * 3600, n).round()
end = np.where(status == "Running", np.nan, end)          # still running: no end time yet

jobs_src = pd.DataFrame({
    "job_id": [f"job_{i:04d}" for i in range(n)],
    "user": rng.choice([f"user_{u:02d}" for u in range(25)], n),
    "status": status,
    "gpu_type": rng.choice(["V100", "T4", "A10", "P100"], n, p=[0.45, 0.30, 0.15, 0.10]),
    "plan_gpu": rng.choice([25, 50, 100, 200, 400, 800], n, p=[0.10, 0.15, 0.35, 0.20, 0.12, 0.08]),
    "submit_time": submit, "start_time": start, "end_time": end,
})

# --- break it on purpose ---
perm = rng.permutation(n)
messy = perm[:40]                                        # 40 labels with odd case / stray spaces
jobs_src.loc[messy, "gpu_type"] = [rng.choice([s.lower(), f" {s}", f"{s} "]) for s in jobs_src.loc[messy, "gpu_type"]]
jobs_src.loc[perm[40:55], "gpu_type"] = np.nan           # 15 missing labels
rest = jobs_src.loc[perm[55:]]
jobs_src.loc[rest[rest["gpu_type"] == "V100"].index[:10], "gpu_type"] = "Tesla-V100"    # an alias
done = rest[rest["status"] != "Running"].index[-8:]
jobs_src.loc[done, "end_time"] = jobs_src.loc[done, "start_time"] - rng.integers(60, 3600, 8)  # end before start
jobs_src.loc[perm[200:205], "plan_gpu"] = 0              # jobs asking for zero GPUs
dupes = jobs_src.sample(12, random_state=3)              # 12 rows exported twice
jobs_src = pd.concat([jobs_src, dupes]).sample(frac=1, random_state=4).reset_index(drop=True)
jobs_src.to_csv("trace_jobs.csv", index=False)

# lookup tables
pd.DataFrame({"gpu_type": ["V100", "T4", "P100", "T4"],          # note: T4 pasted twice, A10 missing
              "memory_gb": [32, 16, 16, 16],
              "tdp_w": [300, 70, 250, 70]}).to_csv("gpu_specs.csv", index=False)
pd.DataFrame({"gpu_type": ["V100", "T4", "A10", "P100"],
              "installed_gpus": [400, 200, 120, 80]}).to_csv("gpu_capacity.csv", index=False)
print("saved trace_jobs.csv, gpu_specs.csv, gpu_capacity.csv")

saved trace_jobs.csv, gpu_specs.csv, gpu_capacity.csv


---
## Part 1: First look. Profile before you clean

The first 5 minutes with any new dataset are the same: how many rows, what types, what's missing, what values appear. In SQL terms:

| SQL | pandas |
|---|---|
| `SELECT COUNT(*)` | `len(df)` |
| `COUNT(*) - COUNT(col)` | `df["col"].isna().sum()` |
| `COUNT(DISTINCT col)` | `df["col"].nunique()` |
| `SELECT col, COUNT(*) ... GROUP BY col` | `df["col"].value_counts(dropna=False)` |

In [ ]:
raw = pd.read_csv("trace_jobs.csv")
print(raw.shape)
raw.info()

(612, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   job_id       612 non-null    object 
 1   user         612 non-null    object 
 2   status       612 non-null    object 
 3   gpu_type     597 non-null    object 
 4   plan_gpu     612 non-null    int64  
 5   submit_time  612 non-null    int64  
 6   start_time   612 non-null    float64
 7   end_time     565 non-null    float64
dtypes: float64(2), int64(2), object(4)
memory usage: 38.4+ KB


In [ ]:
print(raw.isna().sum(), "\n")
print("rows:", len(raw), "| distinct job_ids:", raw["job_id"].nunique(), "\n")
raw["gpu_type"].value_counts(dropna=False)

job_id          0
user            0
status          0
gpu_type       15
plan_gpu        0
submit_time     0
start_time      0
end_time       47
dtype: int64 

rows: 612 | distinct job_ids: 600 



,count
gpu_type,
V100,236
T4,172
A10,80
P100,59
NaN,15
Tesla-V100,10
V100,9
V100,8
v100,4


Before reading on, list every problem you can see. You should spot at least four: more rows than distinct job IDs, GPU labels that differ only by case or spaces, a `Tesla-V100` alias, missing `gpu_type`, and missing `end_time`.

`dropna=False` matters: without it, `value_counts` hides the missing values, which is exactly what you're trying to find. Keep this list. It becomes the "data issues" section of your README.

---
## Part 2: Duplicates

In [ ]:
print("exact duplicate rows:", raw.duplicated().sum())
raw[raw["job_id"].duplicated(keep=False)].sort_values("job_id").head(6)

exact duplicate rows: 12


,job_id,user,status,gpu_type,plan_gpu,submit_time,start_time,end_time
501,job_0010,user_03,Terminated,V100,200,245085,245156.0,256871.0
209,job_0010,user_03,Terminated,V100,200,245085,245156.0,256871.0
417,job_0037,user_15,Terminated,V100,200,866618,867938.0,868627.0
367,job_0037,user_15,Terminated,V100,200,866618,867938.0,868627.0
9,job_0053,user_12,Terminated,T4,100,307904,308557.0,313424.0
24,job_0053,user_12,Terminated,T4,100,307904,308557.0,313424.0


In [ ]:
jobs = raw.drop_duplicates().copy()          # SELECT DISTINCT *
print(len(raw), "→", len(jobs), "rows | job_ids still duplicated:", jobs["job_id"].duplicated().sum())

612 → 600 rows | job_ids still duplicated: 0


These were **exact** copies, so `drop_duplicates()` is safe. If the same `job_id` appeared twice with *different* values (say, two end times), you'd need a rule, like "keep the latest record", and you'd write that rule down. Always re-check uniqueness after de-duplicating; don't assume it worked.

`.copy()` tells pandas you want a separate table, not a view of `raw`. It avoids a confusing `SettingWithCopyWarning` later.

---
## Part 3: Messy text labels

`.str` unlocks string functions on a column, the same way `.dt` unlocked dates on Day 2.

In [ ]:
before = jobs["gpu_type"].copy()                                   # keep the original to compare later

jobs["gpu_type"] = jobs["gpu_type"].str.strip().str.upper()        # TRIM(UPPER(gpu_type))
jobs["gpu_type"] = jobs["gpu_type"].replace({"TESLA-V100": "V100"})   # CASE WHEN ... alias mapping
jobs["gpu_type"].value_counts(dropna=False)

,count
gpu_type,
V100,259
T4,176
A10,87
P100,63
NaN,15


Five clean labels plus the missing ones. A mapping dictionary like `{"TESLA-V100": "V100"}` is how you'll handle aliases in the real trace: easy to read, easy to extend.

✏️ **Exercise 1.** How many rows had their label changed by the cleaning? Compare `before` with `jobs["gpu_type"]`. Try `(before != jobs["gpu_type"]).sum()` first, then look closely at the answer. Is it too high? Why? (Hint: in SQL, is `NULL = NULL` true?)

In [ ]:
# ✏️ Exercise 1: your code here
(before != jobs["gpu_type"]).sum()

np.int64(65)

---
## Part 4: Missing values, units and GPU-hours

Missing values need a **decision per column**, not one blanket `dropna()`.

**`gpu_type` missing:** the job still used GPUs, so dropping it would understate demand. Label it instead.

In [ ]:
jobs["gpu_type"] = jobs["gpu_type"].fillna("UNKNOWN")      # COALESCE(gpu_type, 'UNKNOWN')

**`end_time` missing:** before deciding, check *which* rows are missing it. `pd.crosstab` is a quick two-way count, like `GROUP BY status, end_time IS NULL`.

In [ ]:
pd.crosstab(jobs["status"], jobs["end_time"].isna())

end_time,False,True
status,,
Failed,73,0
Running,0,45
Terminated,482,0


Every missing end time is a **Running** job. That's not broken data: the job simply hadn't finished when the trace stopped. Statisticians call this **right-censored**. If you dropped these rows, you'd drop the most recent demand, which is exactly the demand a forecast cares about most.

The usual fix: count their usage **up to the end of the trace**, in a new column so the original stays untouched. Then convert units and compute GPU-hours (Day 2).

In [ ]:
trace_start = pd.Timestamp("2026-03-04")                          # assumed anchor date, as on Day 2
trace_end_s = jobs[["start_time", "end_time"]].max().max()        # last moment the trace saw anything

jobs["end_time_filled"] = jobs["end_time"].fillna(trace_end_s)   # running jobs: usage so far
jobs["gpus"] = jobs["plan_gpu"] / 100                             # percent of a GPU → GPUs
jobs["duration_s"] = jobs["end_time_filled"] - jobs["start_time"]
jobs["gpu_hours"] = jobs["gpus"] * jobs["duration_s"] / 3600
jobs["wait_s"] = jobs["start_time"] - jobs["submit_time"]         # time spent in the queue
jobs["submit_ts"] = trace_start + pd.to_timedelta(jobs["submit_time"], unit="s")

jobs[["job_id", "status", "gpu_type", "gpus", "duration_s", "gpu_hours"]].head()

,job_id,status,gpu_type,gpus,duration_s,gpu_hours
0,job_0359,Terminated,T4,1.00,10972.0,3.047778
1,job_0203,Terminated,A10,0.25,22195.0,1.541319
2,job_0180,Terminated,T4,0.50,9903.0,1.375417
3,job_0519,Running,V100,0.25,49734.0,3.453750
4,job_0521,Running,V100,1.00,45637.0,12.676944


**Unit mistakes are the most expensive bugs in capacity planning.** If you forgot the `/ 100`, every number downstream would be 100× too big and still look plausible on a chart. Write the unit into the column name (`_s`, `_hours`, `_w`) whenever you can.

✏️ **Exercise 2.** What share of total GPU-hours comes from `Running` jobs? If you had dropped rows with a missing `end_time`, how much demand would you have missed? (Numbers shift slightly after Part 5 removes bad rows; that's fine.)

In [ ]:
# ✏️ Exercise 2: your code here

jobs.groupby("status")["gpu_hours"].sum()/jobs["gpu_hours"].sum()

,gpu_hours
status,
Failed,0.089936
Running,0.331581
Terminated,0.578484


---
## Part 5: Flag bad rows instead of deleting them

Some rows are impossible: a job that ends before it starts, or one that asks for zero GPUs. Rather than deleting them in place, **label** each row with its issue, then filter. You keep a record of what you dropped and why, which you can show in the README and defend in an interview.

`np.select` is pandas' `CASE WHEN`: the first true condition wins.

In [ ]:
conditions = [
    jobs["duration_s"] < 0,          # WHEN duration_s < 0 THEN 'negative_duration'
    jobs["gpus"] <= 0,               # WHEN gpus <= 0      THEN 'zero_gpus'
]
jobs["issue"] = np.select(conditions, ["negative_duration", "zero_gpus"], default="ok")
jobs["issue"].value_counts()

,count
issue,
ok,587
negative_duration,8
zero_gpus,5


In [ ]:
dropped = jobs[jobs["issue"] != "ok"]
clean = jobs[jobs["issue"] == "ok"].copy()
print(f"kept {len(clean)} of {len(jobs)} jobs; dropped {len(dropped)}")
dropped[["job_id", "status", "gpu_type", "gpus", "duration_s", "issue"]]
printf(clean)

kept 587 of 600 jobs; dropped 13


,job_id,user,status,gpu_type,plan_gpu,submit_time,start_time,end_time,end_time_filled,gpus,duration_s,gpu_hours,wait_s,submit_ts,issue
0,job_0359,user_03,Terminated,T4,100,800961,802870.0,813842.0,813842.0,1.00,10972.0,3.047778,1909.0,2026-03-13 06:29:21,ok
1,job_0203,user_04,Terminated,A10,25,701035,701182.0,723377.0,723377.0,0.25,22195.0,1.541319,147.0,2026-03-12 02:43:55,ok
2,job_0180,user_00,Terminated,T4,50,1207711,1207793.0,1217696.0,1217696.0,0.50,9903.0,1.375417,82.0,2026-03-17 23:28:31,ok
3,job_0519,user_09,Running,V100,25,1197214,1199547.0,NaN,1249281.0,0.25,49734.0,3.453750,2333.0,2026-03-17 20:33:34,ok
4,job_0521,user_02,Running,V100,100,1203624,1203644.0,NaN,1249281.0,1.00,45637.0,12.676944,20.0,2026-03-17 22:20:24,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
607,job_0599,user_13,Terminated,T4,50,28957,29286.0,78076.0,78076.0,0.50,48790.0,6.776389,329.0,2026-03-04 08:02:37,ok
608,job_0360,user_05,Terminated,V100,50,298429,298567.0,306416.0,306416.0,0.50,7849.0,1.090139,138.0,2026-03-07 10:53:49,ok
609,job_0439,user_07,Terminated,T4,200,118955,118981.0,132771.0,132771.0,2.00,13790.0,7.661111,26.0,2026-03-05 09:02:35,ok
610,job_0174,user_15,Terminated,P100,100,702335,702545.0,705413.0,705413.0,1.00,2868.0,0.796667,210.0,2026-03-12 03:05:35,ok


✏️ **Exercise 3.** Some jobs waited more than **2 hours** between submit and start. Count them in `clean` and find their share of GPU-hours. Then decide: are these bad rows to drop, or something else? Write one sentence on why.

In [ ]:
# ✏️ Exercise 3: your code here
[clean["wait_s"]>=7200].count()/clean["wait_s"].count()

23 jobs waited > 2 hours (3.9% of all jobs).
These jobs account for 5.1% of total GPU-hours.


---
## Part 6: Merges without silent damage

You did a basic `merge` on Day 1. With real lookup tables, three things go wrong, and none of them raises an error by default.

| SQL | pandas |
|---|---|
| `LEFT JOIN specs USING (gpu_type)` | `clean.merge(specs, on="gpu_type", how="left")` |
| `... WHERE specs.gpu_type IS NULL` (anti-join) | `indicator=True`, then filter `_merge == "left_only"` |
| `UNION ALL` | `pd.concat([a, b], ignore_index=True)` |

### 6.1 Fan-out: the join that quietly multiplies rows

In [ ]:
specs = pd.read_csv("gpu_specs.csv")
specs

,gpu_type,memory_gb,tdp_w
0,V100,32,300
1,T4,16,70
2,P100,16,250
3,T4,16,70


In [ ]:
merged = clean.merge(specs, on="gpu_type", how="left")
print(len(clean), "→", len(merged), "rows")
print(f"GPU-hours: {clean['gpu_hours'].sum():,.0f} → {merged['gpu_hours'].sum():,.0f}")

587 → 587 rows
GPU-hours: 5,987 → 5,987


The row count went **up**. `T4` appears twice in `specs`, so every T4 job matched twice and T4 demand doubled. Nothing errored, and a chart built on `merged` would look perfectly normal. This is called **fan-out**, and it's the same bug as a SQL join on a non-unique key.

### 6.2 `validate`: make pandas refuse a bad join
The next cell is **meant to fail**. Read the last line of the error, as on Day 1.

In [ ]:
clean.merge(specs, on="gpu_type", how="left", validate="many_to_one")

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

`validate="many_to_one"` means "many jobs can share one spec, but each spec must be unique". When that's false, pandas stops you. Make it a habit on every lookup join. Now fix the lookup table and add `indicator=True` to see what matched:

In [ ]:
specs = specs.drop_duplicates("gpu_type")
merged = clean.merge(specs, on="gpu_type", how="left", validate="many_to_one", indicator=True)
print(len(clean), "→", len(merged), "rows")
merged["_merge"].value_counts()

587 → 587 rows


,count
_merge,
both,488
left_only,99
right_only,0


### 6.3 Anti-join: which jobs found no match?

In [ ]:
merged.loc[merged["_merge"] == "left_only", "gpu_type"].value_counts()

,count
gpu_type,
A10,84
UNKNOWN,15


`UNKNOWN` is expected. `A10` is a real gap: the spec table just doesn't have it. With a plain left join those jobs would get blank power figures, and a later `sum()` would quietly skip them. Add the missing row (UNION ALL) and re-run the join:

In [ ]:
a10 = pd.DataFrame({"gpu_type": ["A10"], "memory_gb": [24], "tdp_w": [150]})
specs = pd.concat([specs, a10], ignore_index=True)        # UNION ALL

merged = clean.merge(specs, on="gpu_type", how="left", validate="many_to_one", indicator=True)
merged.loc[merged["_merge"] == "left_only", "gpu_type"].value_counts()

,count
gpu_type,
UNKNOWN,15


✏️ **Exercise 4.** A first step from GPUs toward megawatts: compute energy per job as `kwh = gpu_hours × tdp_w / 1000`, then total GPU-hours and kWh **by `gpu_type`**. Does the type with the most GPU-hours also use the most energy? What happens to the `UNKNOWN` jobs in the kWh total, and why does that matter?

In [ ]:
# ✏️ Exercise 4: your code here

merged.kwh = merged.gpu_hours * merged.tdp_w / 1000
energy.total_gpu_hours = merged.groupby("gpu_type")("kwh","gpu_hours").sum().sort_values("kwh", ascending=False)
energy.total_gpu_hours



,gpu_hours,kwh
gpu_type,,
V100,2707.694722,812.308417
A10,959.397847,143.909677
P100,475.569375,118.892344
T4,1598.156528,111.870957
UNKNOWN,245.932917,0.000000


---
## Part 7: A reusable sanity-check function

Every cleaning step above could have gone wrong silently. Instead of eyeballing each time, write the checks once and run them after every change. This function is the start of a `checks.py` you'll keep in the repo and run on the real trace.

In [ ]:
capacity = pd.read_csv("gpu_capacity.csv")

def run_checks(raw, clean, dropped, merged, capacity):
    """Return a table of pass/fail checks on the cleaned trace."""
    results = []
    def check(name, passed, detail=""):
        results.append({"check": name, "passed": bool(passed), "detail": detail})

    n_dupes = raw.duplicated().sum()
    check("rows reconcile (raw = clean + dupes + dropped)",
          len(raw) == len(clean) + n_dupes + len(dropped),
          f"{len(raw)} = {len(clean)} + {n_dupes} + {len(dropped)}")
    check("job_id is unique", clean["job_id"].is_unique)
    check("no negative durations", (clean["duration_s"] >= 0).all())
    check("merge kept row count", len(merged) == len(clean), f"{len(clean)} → {len(merged)}")
    check("merge kept GPU-hours",
          np.isclose(merged["gpu_hours"].sum(), clean["gpu_hours"].sum()),
          f"{clean['gpu_hours'].sum():,.0f} → {merged['gpu_hours'].sum():,.0f}")
    unmatched = sorted(set(merged.loc[merged["tdp_w"].isna(), "gpu_type"]))
    check("every known GPU type has specs", set(unmatched) <= {"UNKNOWN"}, ", ".join(unmatched))

    daily = (clean.assign(day=clean["submit_ts"].dt.floor("D"))
                  .groupby(["gpu_type", "day"], as_index=False)["gpu_hours"].sum()
                  .merge(capacity, on="gpu_type", how="inner"))
    over = daily[daily["gpu_hours"] > daily["installed_gpus"] * 24]
    check("daily GPU-hours fit installed capacity", over.empty, f"{len(over)} type-days over")

    return pd.DataFrame(results)

run_checks(raw, clean, dropped, merged, capacity)

,check,passed,detail
0,rows reconcile (raw = clean + dupes + dropped),True,612 = 587 + 12 + 13
1,job_id is unique,True,
2,no negative durations,True,
3,merge kept row count,True,587 → 587
4,merge kept GPU-hours,True,"5,987 → 5,987"
5,every known GPU type has specs,True,UNKNOWN
6,daily GPU-hours fit installed capacity,True,0 type-days over


All green. Each check guards one mistake from today:

- **Reconciliation:** every raw row is accounted for (kept, de-duplicated, or dropped with a reason). Finance people love this one.
- **Fan-out guards:** row count and GPU-hours unchanged by the join.
- **Coverage:** no real GPU type is missing specs.
- **Physical limit:** you can't use more GPU-hours in a day than `installed GPUs × 24`. If this ever fails, the data or a unit is wrong, not the fleet. (It uses the Day 2 simplification of booking hours on the submit day.)

✏️ **Exercise 5 (stretch).** Break it on purpose: rebuild a bad merge from the original `gpu_specs.csv` (with the duplicate T4 and no A10), without `validate`, and pass it to `run_checks` in place of `merged`. Which checks fail? Do the failures point you straight at the problems?

In [ ]:
# ✏️ Exercise 5: your code here
bad_merge = clean.merge(specs, on="gpu_type", how="left")
run_checks(raw, clean, dropped, bad_merge, capacity)

,check,passed,detail
0,rows reconcile (raw = clean + dupes + dropped),True,612 = 587 + 12 + 13
1,job_id is unique,True,
2,no negative durations,True,
3,merge kept row count,False,587 → 761
4,merge kept GPU-hours,False,"5,987 → 7,585"
5,every known GPU type has specs,False,"A10, UNKNOWN"
6,daily GPU-hours fit installed capacity,True,0 type-days over


---
## Part 8: The four questions

Imagine Claude Code wrote this block. Answer before you run it:
1. What is the input?
2. What transformation happens?
3. What is the output?
4. How would I know if the result is wrong?

In [ ]:
by_type = (
    merged.groupby("gpu_type", as_index=False)
          .agg(jobs=("job_id", "count"), gpu_hours=("gpu_hours", "sum"))
          .assign(share=lambda d: d["gpu_hours"] / d["gpu_hours"].sum())
          .sort_values("gpu_hours", ascending=False)
)
by_type

,gpu_type,jobs,gpu_hours,share
4,V100,253,2707.694722,0.452281
2,T4,174,1598.156528,0.266949
0,A10,84,959.397847,0.160253
1,P100,61,475.569375,0.079437
3,UNKNOWN,15,245.932917,0.041080


✏️ **Your answers** (double-click to edit). One new piece: `.assign(share=lambda d: ...)` adds a column in the middle of a chain, where `d` is the table at that point.

1. Input:merged dataframe that is aggreagated by gpu_type
2. Transformation: aggregated & then values such as count of jobs, total gpu_hours and fraction of gpu_hours are calucalted
3. Output: seen above
4. How I'd spot a wrong result:

---
## Wrap-up

**Add to `glossary.md` today, in your own words:**
- **Right-censored job:** still running when the trace ends; count its usage up to the trace end, don't drop it
- **Fan-out:** a join on a non-unique key that silently duplicates rows and inflates totals
- **Anti-join:** rows in one table with no match in another (`LEFT JOIN ... WHERE right.key IS NULL`)
- **Reconciliation:** proving every input row is kept, de-duplicated, or dropped with a reason

**Done today when you can:**
- [ ] Explain why dropping rows with a missing `end_time` would understate recent demand
- [ ] Say what `validate="many_to_one"` protects against, in one sentence
- [ ] Name the four checks you'd run first on any new trace

**Next (Day 4):** open the real Alibaba v2026 trace documentation, map its columns and units to what you cleaned today, and list the issues you expect to find before loading a single row.

---
---
## Solutions (try first!)

In [ ]:
# Exercise 1
print("naive count:", (before != jobs["gpu_type"]).sum())         # too high: NaN != NaN is True
changed = before.ne(jobs["gpu_type"]) & before.notna()
print("labels actually changed:", changed.sum())
# Like SQL, a missing value never equals anything, not even another missing value.
# (By this point jobs["gpu_type"] has UNKNOWN filled in; the notna() filter still gives the right count.)

In [ ]:
# Exercise 2
share = jobs.groupby("status")["gpu_hours"].sum() / jobs["gpu_hours"].sum()
print(share.map("{:.1%}".format))
# Running jobs are under 8% of jobs but about a third of GPU-hours here: they're the newest work and are
# counted all the way to the trace end. Dropping them would hide a big slice of the most recent demand.

In [ ]:
# Exercise 3
long_queue = clean[clean["wait_s"] > 2 * 3600]
print(len(long_queue), "jobs waited > 2h;",
      f"{long_queue['gpu_hours'].sum() / clean['gpu_hours'].sum():.1%} of GPU-hours")
# Keep them. A long queue isn't bad data: it's a sign demand exceeded available capacity at that moment.
# Queue time is a capacity signal in its own right (unmet demand), so flag it, don't drop it.

23 jobs waited > 2h; 5.1% of GPU-hours


In [ ]:
# Exercise 4
merged["kwh"] = merged["gpu_hours"] * merged["tdp_w"] / 1000
energy = merged.groupby("gpu_type")[["gpu_hours", "kwh"]].sum().sort_values("kwh", ascending=False)
print(energy)
print("UNKNOWN kWh is 0 because tdp_w is missing and sum() skips NaN:",
      merged.loc[merged["gpu_type"] == "UNKNOWN", "kwh"].isna().all())
# Low-power T4s can rank high on GPU-hours but low on kWh. And the UNKNOWN jobs vanish from the energy
# total without any error: a silent gap you'd want to call out or estimate (e.g. fleet-average TDP).

In [ ]:
# Exercise 5
# Fails: "merge kept row count" and "merge kept GPU-hours" (the duplicate T4),
# and "every known GPU type has specs" (A10). Each failure names the problem.
bad = clean.merge(pd.read_csv("gpu_specs.csv"), on="gpu_type", how="left")
run_checks(raw, clean, dropped, bad, capacity)